# Stability

In [5]:
import pandas as pd
import numpy as np
from functions.running import prepare_data
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.decomposition import PCA
from functions.training import train
from functions.networks import SimpleNN, FullNN
import shap

In [ ]:
# Convert KernelSHAP output (list of class-wise arrays)
# to shape (n_samples, n_features)
def combine_shap(shap_list, y_true):
    shap_arr = []
    class_counters = [0] * len(shap_list)  # track position in each class's SHAP array

    for c in y_true:
        idx = class_counters[c]             # current index for this class
        shap_arr.append(shap_list[c][idx])
        class_counters[c] += 1              # move to next sample in that class

    return np.vstack(shap_arr)


def explanation_variance_torch(attributions, signed=False, device=None):
    """
    Compute σ²_exp stability measure using PyTorch.
    
    attributions: Tensor (n_runs, n_samples, n_features)
    signed: If False, use absolute attributions |e|
    device: "cpu" or "cuda"
    
    Returns:
        variance_per_sample: Tensor (n_samples,)
        mean_variance: float
    """

    # Move to device
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    
    A = attributions.to(device).float()  # shape (N, S, F)
    
    if not signed:
        A = A.abs()

    # Mean across runs
    mean_A = A.mean(dim=0, keepdim=True)   # shape (1, S, F)

    # σ²_exp = mean( (e_i - ē)^2 ) across runs
    var_run = ((A - mean_A) ** 2).mean(dim=0)   # shape (S, F)

    # Reduce to per-sample variance (mean across features)
    variance_per_sample = var_run.mean(dim=1)   # shape (S,)

    # Global variance score
    mean_variance = variance_per_sample.mean().item()

    return variance_per_sample.cpu(), mean_variance



In [7]:
df_train = pd.read_table('https://archive.ics.uci.edu/ml/machine-learning-databases/spect/SPECTF.train', header = None,sep=',')
df_test = pd.read_table('https://archive.ics.uci.edu/ml/machine-learning-databases/spect/SPECTF.test',
                     header=None, sep = ',')
df = pd.concat([df_train, df_test])
data = df.to_numpy()
X,y = data[:,1:], data[:,0]
G = len(np.unique(y))
for g in range(G):
  print(sum(y==g))
X = X.astype('float')

dataname = 'heart'
input_dim = X.shape[1]
epochs = 200
batch_size = 64
n_layer = 5
init_type = 'he'
dataname_ = dataname + str(n_layer) + init_type
missing = False
variance_retained = .95
criterion = nn.CrossEntropyLoss()
learning_rate = 0.01
n_frozen_epochs = 30

55
212


### PCsInit

In [ ]:
n_seeds = 10
random_states = np.random.choice(range(1000, 2000), size=n_seeds, replace=False)
print("Random states:", random_states)

all_results = []

for random_state in random_states: 
    X_train, X_test, y_train, y_test = prepare_data(X,y, missing = missing, random_state=1431)
    input_dim = X_train.shape[1]  # Number of features

    output_dim = len(np.unique(y_train))  # Number of classes (for Iris dataset)

    pca = PCA(n_components=variance_retained)
    pca.fit(X_train)
    n_components = pca.n_components_

    hidden_dim = n_components  # Hidden layer size

    other_layers = SimpleNN(input_dim=n_components, hidden_dim=hidden_dim, output_dim=output_dim, n_layer = n_layer, init_type = init_type)

    train_loader = torch.utils.data.DataLoader(list(zip(X_train, y_train)), batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(list(zip(X_test, y_test)), batch_size=batch_size, shuffle=False)


    print("Training with PCA-initialized NN...")
    pca_init_nn = FullNN(input_dim, n_components, other_layers, activation='none', init_type=init_type)
    pca_init_nn.init_pca_weights(X_train)  # Initialize weights with PCA components

    # train on everything except the first layer
    optimizer = optim.Adam([{'params': param} for name, param in pca_init_nn.named_parameters() if not name.startswith('fc1')],
                            lr=learning_rate)

    model_pcsinit, train_losses_pcinit, test_accuracies_pcinit, training_time_pcinit, test_probs_pcinit = train(
            pca_init_nn, train_loader, test_loader, criterion, optimizer, epochs=n_frozen_epochs
        )


    # train the complete network
    optimizer = optim.Adam(pca_init_nn.parameters(), lr=learning_rate)
    model_pcsinit2, train_losses_pcinit2, test_accuracies_pcinit2, training_time_pcinit2, test_probs_pcinit2 = train(pca_init_nn, train_loader, test_loader, criterion, optimizer, epochs=epochs-n_frozen_epochs)
    train_losses_pcinit = np.concatenate([train_losses_pcinit,train_losses_pcinit2])
    test_accuracies_pcinit = np.concatenate([test_accuracies_pcinit, test_accuracies_pcinit2])
    training_time_pcinit = np.concatenate([training_time_pcinit, training_time_pcinit2])
    test_probs_pcinit = np.concatenate([test_probs_pcinit, test_probs_pcinit2])

    # ====== Interpret Full Model After Training ======
    # --- 1. Model cuối cùng ---
    model_final = model_pcsinit2  # model đã fine-tune cả fc1
    model_final.eval()

    X_train_np = X_train.numpy() if hasattr(X_train, 'numpy') else X_train
    X_test_np  = X_test.numpy()  if hasattr(X_test, 'numpy') else X_test
    y_test_np  = y_test.numpy()  if hasattr(y_test, 'numpy') else y_test

    # -----------------------------
    # Step 1: Create balanced test set for SHAP
    # -----------------------------
    idx_class0 = np.where(y_test_np == 0)[0]
    idx_class1 = np.where(y_test_np == 1)[0]
    n_samples = min(len(idx_class0), len(idx_class1))  # balanced size

    np.random.seed(42)  # reproducible
    idx_class0_sample = np.random.choice(idx_class0, n_samples, replace=False)
    idx_class1_sample = np.random.choice(idx_class1, n_samples, replace=False)

    balanced_idx = np.concatenate([idx_class0_sample, idx_class1_sample])
    np.random.shuffle(balanced_idx)

    test_np_balanced = X_test_np[balanced_idx]
    y_test_balanced  = y_test_np[balanced_idx]

    print(f"Balanced test set shape: {test_np_balanced.shape}")
    print(f"Class counts: {np.bincount(y_test_balanced)}")  # should be [n_samples, n_samples]

    print("\n=== SHAP PCSINIT ===")
    background_np = X_train_np
    #test_np_20 = test_np_balanced.numpy()[:20]

    def pcsinit_torch_predict(x_numpy):
        x_torch = torch.tensor(x_numpy, dtype=torch.float32)
        with torch.no_grad():
            return model_final(x_torch).numpy()

    pcsinit_explainer = shap.KernelExplainer(pcsinit_torch_predict, background_np)
    pcsinit_shap_values = pcsinit_explainer.shap_values(test_np_balanced)
    combined_shap = combine_shap(pcsinit_shap_values, y_test_balanced)
    all_results.append(combined_shap)

# After loop
shap_tensor = torch.tensor(np.stack(all_results, axis=0))
variance_per_sample, mean_variance = explanation_variance_torch(shap_tensor)

print("Stability σ²_exp:", mean_variance)


Random states: [1212, 1462, 1512, 1431, 1651, 1013, 1690, 1515, 1997, 1549]
Training with PCA-initialized NN...
Number of PCA components: 24
Epoch 1/30, Training Loss: 0.6089, Testing Accuracy: 0.8395, Training Time: 0.0185
Epoch 2/30, Training Loss: 0.5194, Testing Accuracy: 0.8395, Training Time: 0.0293
Epoch 3/30, Training Loss: 0.4411, Testing Accuracy: 0.8025, Training Time: 0.0383
Epoch 4/30, Training Loss: 0.4267, Testing Accuracy: 0.8148, Training Time: 0.0501
Epoch 5/30, Training Loss: 0.3939, Testing Accuracy: 0.8395, Training Time: 0.0605
Epoch 6/30, Training Loss: 0.3789, Testing Accuracy: 0.8272, Training Time: 0.0705
Epoch 7/30, Training Loss: 0.3407, Testing Accuracy: 0.8025, Training Time: 0.0812
Epoch 8/30, Training Loss: 0.3145, Testing Accuracy: 0.7901, Training Time: 0.0911
Epoch 9/30, Training Loss: 0.2880, Testing Accuracy: 0.7901, Training Time: 0.1014
Epoch 10/30, Training Loss: 0.2481, Testing Accuracy: 0.7160, Training Time: 0.1083
Epoch 11/30, Training Loss: 

Using 186 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


Epoch 156/170, Training Loss: 0.0000, Testing Accuracy: 0.7654, Training Time: 1.3095
Epoch 157/170, Training Loss: 0.0000, Testing Accuracy: 0.7654, Training Time: 1.3200
Epoch 158/170, Training Loss: 0.0000, Testing Accuracy: 0.7654, Training Time: 1.3282
Epoch 159/170, Training Loss: 0.0000, Testing Accuracy: 0.7654, Training Time: 1.3358
Epoch 160/170, Training Loss: 0.0000, Testing Accuracy: 0.7654, Training Time: 1.3446
Epoch 161/170, Training Loss: 0.0000, Testing Accuracy: 0.7654, Training Time: 1.3533
Epoch 162/170, Training Loss: 0.0000, Testing Accuracy: 0.7654, Training Time: 1.3619
Epoch 163/170, Training Loss: 0.0000, Testing Accuracy: 0.7654, Training Time: 1.3720
Epoch 164/170, Training Loss: 0.0000, Testing Accuracy: 0.7654, Training Time: 1.3804
Epoch 165/170, Training Loss: 0.0000, Testing Accuracy: 0.7654, Training Time: 1.3882
Epoch 166/170, Training Loss: 0.0000, Testing Accuracy: 0.7654, Training Time: 1.3969
Epoch 167/170, Training Loss: 0.0000, Testing Accuracy

  0%|          | 0/26 [00:00<?, ?it/s]

Training with PCA-initialized NN...
Number of PCA components: 24
Epoch 1/30, Training Loss: 0.5975, Testing Accuracy: 0.8395, Training Time: 0.0107
Epoch 2/30, Training Loss: 0.5326, Testing Accuracy: 0.8395, Training Time: 0.0216
Epoch 3/30, Training Loss: 0.4587, Testing Accuracy: 0.8395, Training Time: 0.0316
Epoch 4/30, Training Loss: 0.4480, Testing Accuracy: 0.8272, Training Time: 0.0413
Epoch 5/30, Training Loss: 0.4260, Testing Accuracy: 0.8272, Training Time: 0.0516
Epoch 6/30, Training Loss: 0.4050, Testing Accuracy: 0.8272, Training Time: 0.0619
Epoch 7/30, Training Loss: 0.3727, Testing Accuracy: 0.7531, Training Time: 0.0719
Epoch 8/30, Training Loss: 0.3420, Testing Accuracy: 0.7407, Training Time: 0.0819
Epoch 9/30, Training Loss: 0.3115, Testing Accuracy: 0.7778, Training Time: 0.0913
Epoch 10/30, Training Loss: 0.2999, Testing Accuracy: 0.7407, Training Time: 0.1009
Epoch 11/30, Training Loss: 0.2594, Testing Accuracy: 0.7160, Training Time: 0.1079
Epoch 12/30, Trainin

...

Training with PCA-initialized NN...
Number of PCA components: 24
Epoch 1/30, Training Loss: 0.5964, Testing Accuracy: 0.8395, Training Time: 0.0100
Epoch 2/30, Training Loss: 0.4651, Testing Accuracy: 0.8395, Training Time: 0.0194
Epoch 3/30, Training Loss: 0.4262, Testing Accuracy: 0.8395, Training Time: 0.0292
Epoch 4/30, Training Loss: 0.4033, Testing Accuracy: 0.8395, Training Time: 0.0392
Epoch 5/30, Training Loss: 0.3761, Testing Accuracy: 0.8395, Training Time: 0.0502
Epoch 6/30, Training Loss: 0.3551, Testing Accuracy: 0.8272, Training Time: 0.0607
Epoch 7/30, Training Loss: 0.3186, Testing Accuracy: 0.7901, Training Time: 0.0704
Epoch 8/30, Training Loss: 0.2796, Testing Accuracy: 0.7778, Training Time: 0.0809
Epoch 9/30, Training Loss: 0.2432, Testing Accuracy: 0.8025, Training Time: 0.0905
Epoch 10/30, Training Loss: 0.2098, Testing Accuracy: 0.7654, Training Time: 0.1003
Epoch 11/30, Training Loss: 0.1726, Testing Accuracy: 0.7407, Training Time: 0.1078
Epoch 12/30, Trainin

Using 186 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


Epoch 166/170, Training Loss: 0.0000, Testing Accuracy: 0.7531, Training Time: 1.1376
Epoch 167/170, Training Loss: 0.0000, Testing Accuracy: 0.7531, Training Time: 1.1450
Epoch 168/170, Training Loss: 0.0000, Testing Accuracy: 0.7531, Training Time: 1.1539
Epoch 169/170, Training Loss: 0.0000, Testing Accuracy: 0.7531, Training Time: 1.1598
Epoch 170/170, Training Loss: 0.0000, Testing Accuracy: 0.7531, Training Time: 1.1675
Balanced test set shape: (26, 44)
Class counts: [13 13]

=== SHAP PCSINIT ===


  0%|          | 0/26 [00:00<?, ?it/s]

Stability σ²_exp: 0.6216663122177124


In [25]:
shap_tensor = torch.tensor(np.stack(all_results, axis=0))
variance_per_sample, mean_variance = explanation_variance_torch(shap_tensor)

print("Stability σ²_exp:", mean_variance)

Stability σ²_exp: 0.6216663122177124


### NN


In [ ]:
nn_results = []

for random_state in random_states: 
    X_train, X_test, y_train, y_test = prepare_data(X,y, missing = missing, random_state=1431)
    input_dim = X_train.shape[1]  # Number of features

    output_dim = len(np.unique(y_train))  # Number of classes (for Iris dataset)

    pca = PCA(n_components=variance_retained)
    pca.fit(X_train)
    n_components = pca.n_components_

    hidden_dim = n_components  # Hidden layer size

    other_layers = SimpleNN(input_dim=n_components, hidden_dim=hidden_dim, output_dim=output_dim, n_layer = n_layer, init_type = init_type)

    train_loader = torch.utils.data.DataLoader(list(zip(X_train, y_train)), batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(list(zip(X_test, y_test)), batch_size=batch_size, shuffle=False)

    relu_nn =  FullNN(input_dim, n_components, other_layers, activation='none', init_type=init_type)
    optimizer = optim.Adam(relu_nn.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    nn_model, train_losses_fnn, test_accuracies_fnn, training_time_fnn, _ = train(relu_nn, train_loader, test_loader, criterion, optimizer, epochs=epochs)

    # ====== Interpret Full Model After Training ======
    model_final = nn_model 
    model_final.eval()

    X_train_np = X_train.numpy() if hasattr(X_train, 'numpy') else X_train
    X_test_np  = X_test.numpy()  if hasattr(X_test, 'numpy') else X_test
    y_test_np  = y_test.numpy()  if hasattr(y_test, 'numpy') else y_test

    idx_class0 = np.where(y_test_np == 0)[0]
    idx_class1 = np.where(y_test_np == 1)[0]
    n_samples = min(len(idx_class0), len(idx_class1))  # balanced size

    np.random.seed(42)  # reproducible
    idx_class0_sample = np.random.choice(idx_class0, n_samples, replace=False)
    idx_class1_sample = np.random.choice(idx_class1, n_samples, replace=False)

    balanced_idx = np.concatenate([idx_class0_sample, idx_class1_sample])
    np.random.shuffle(balanced_idx)

    test_np_balanced = X_test_np[balanced_idx]
    y_test_balanced  = y_test_np[balanced_idx]

    print(f"Balanced test set shape: {test_np_balanced.shape}")
    print(f"Class counts: {np.bincount(y_test_balanced)}")  # should be [n_samples, n_samples]

    print("\n=== SHAP NN ===")
    background_np = X_train_np

    def nn_torch_predict(x_numpy):
        x_torch = torch.tensor(x_numpy, dtype=torch.float32)
        with torch.no_grad():
            return nn_model(x_torch).numpy()

    nn_explainer = shap.KernelExplainer(nn_torch_predict, background_np)
    nn_shap_values = nn_explainer.shap_values(test_np_balanced)
    combined_shap = combine_shap(nn_shap_values, y_test_balanced)
    nn_results.append(combined_shap)

# After loop
shap_tensor = torch.tensor(np.stack(nn_results, axis=0))
nn_variance_per_sample, nn_mean_variance = explanation_variance_torch(shap_tensor)

print("Stability σ²_exp:", nn_mean_variance)
#normalized_stability_minmax(shap_tensor)[1]

Epoch 1/200, Training Loss: 0.6531, Testing Accuracy: 0.8395, Training Time: 0.0332
Epoch 2/200, Training Loss: 0.4537, Testing Accuracy: 0.8395, Training Time: 0.0448
Epoch 3/200, Training Loss: 0.3816, Testing Accuracy: 0.8395, Training Time: 0.0556
Epoch 4/200, Training Loss: 0.3378, Testing Accuracy: 0.8395, Training Time: 0.0656
Epoch 5/200, Training Loss: 0.2976, Testing Accuracy: 0.8025, Training Time: 0.0752
Epoch 6/200, Training Loss: 0.2619, Testing Accuracy: 0.8025, Training Time: 0.0830
Epoch 7/200, Training Loss: 0.2216, Testing Accuracy: 0.7778, Training Time: 0.0922
Epoch 8/200, Training Loss: 0.1936, Testing Accuracy: 0.7654, Training Time: 0.0990
Epoch 9/200, Training Loss: 0.1607, Testing Accuracy: 0.7778, Training Time: 0.1054
Epoch 10/200, Training Loss: 0.1331, Testing Accuracy: 0.7654, Training Time: 0.1130
Epoch 11/200, Training Loss: 0.1149, Testing Accuracy: 0.7778, Training Time: 0.1194
Epoch 12/200, Training Loss: 0.0996, Testing Accuracy: 0.7778, Training Ti

Using 186 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


Epoch 198/200, Training Loss: 0.0000, Testing Accuracy: 0.7531, Training Time: 1.8026
Epoch 199/200, Training Loss: 0.0000, Testing Accuracy: 0.7531, Training Time: 1.8132
Epoch 200/200, Training Loss: 0.0000, Testing Accuracy: 0.7531, Training Time: 1.8198
Balanced test set shape: (26, 44)
Class counts: [13 13]

=== SHAP NN ===


  0%|          | 0/26 [00:00<?, ?it/s]

Epoch 1/200, Training Loss: 0.5070, Testing Accuracy: 0.8395, Training Time: 0.0106
Epoch 2/200, Training Loss: 0.3955, Testing Accuracy: 0.8025, Training Time: 0.0200
Epoch 3/200, Training Loss: 0.3243, Testing Accuracy: 0.7407, Training Time: 0.0292
Epoch 4/200, Training Loss: 0.2565, Testing Accuracy: 0.6790, Training Time: 0.0388
Epoch 5/200, Training Loss: 0.2114, Testing Accuracy: 0.7284, Training Time: 0.0474
Epoch 6/200, Training Loss: 0.1517, Testing Accuracy: 0.7037, Training Time: 0.0565
Epoch 7/200, Training Loss: 0.1102, Testing Accuracy: 0.7160, Training Time: 0.0653
Epoch 8/200, Training Loss: 0.0654, Testing Accuracy: 0.7037, Training Time: 0.0750
Epoch 9/200, Training Loss: 0.0306, Testing Accuracy: 0.7037, Training Time: 0.0844
Epoch 10/200, Training Loss: 0.0262, Testing Accuracy: 0.7654, Training Time: 0.0939
Epoch 11/200, Training Loss: 0.0366, Testing Accuracy: 0.7531, Training Time: 0.1021
Epoch 12/200, Training Loss: 0.0373, Testing Accuracy: 0.7284, Training Ti

...

Using 186 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


Epoch 200/200, Training Loss: 0.0000, Testing Accuracy: 0.7901, Training Time: 1.4199
Balanced test set shape: (26, 44)
Class counts: [13 13]

=== SHAP NN ===


  0%|          | 0/26 [00:00<?, ?it/s]

Stability σ²_exp: 1.1301147937774658


0.8618272542953491